# Chapter 3: Fine-Tuning with LoRA

*Small Language Models in Practice — Haji Gul*

> When fine-tuning beats prompting; why LoRA makes it cheap; and a complete,
runnable workflow that adapts a small model to your domain on a single GPU —
data prep, training, and saving the adapter.

---

*Lecture notes mirroring the book. Run the setup cell, then work top-to-bottom. Swap model ids freely.*

## Setup
Uncomment what this chapter needs.

In [ ]:
# %pip install -q transformers datasets accelerate torch
# Chapter-specific installs appear in shell cells below.

## Why and when to fine-tune

Fine-tuning bakes new behavior into the weights. Reach for it when you want the
model to adopt a *style*, a *format*, or *domain reflexes* that a
prompt cannot reliably enforce — consistent JSON output, your support tone, a
narrow classification. (When you instead need fresh *facts*, use retrieval;
that is the next chapter.)

> **Full fine-tuning vs. LoRA.** Full fine-tuning updates every weight — huge memory, easy to overfit on small
data. **LoRA** (Low-Rank Adaptation) freezes the original weights and trains
tiny ``adapter'' matrices instead. You update well under 1% of the parameters,
it fits on a modest GPU, and the result is a small file you can swap in and out.

## Step 1: prepare the data

We will adapt a small model to answer in a strict instruction/response format.
Your dataset just needs an instruction and a target response per row. Here we
load one from the Hub; swap in your own JSON/CSV the same way.

In [ ]:
from datasets import load_dataset

# Any instruction-style dataset works. Replace with your own:
#   load_dataset("json", data_files="my_data.jsonl")
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
dataset = dataset.select(range(2000))   # keep it small for a quick run

def to_text(row):
    instr, ctx, resp = row["instruction"], row["context"], row["response"]
    prompt = instr if not ctx else f"{instr}\n\nContext: {ctx}"
    return {"text": f"### Instruction:\n{prompt}\n\n### Response:\n{resp}"}

dataset = dataset.map(to_text, remove_columns=dataset.column_names)
print(dataset[0]["text"][:300])

## Step 2: load model and tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B"   # base (non-instruct) model to adapt

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
)

## Step 3: attach LoRA adapters

The `peft` library wraps the model so only the adapters train.

In [ ]:
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=16,                 # rank of the adapter (capacity)
    lora_alpha=32,        # scaling
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],   # attention projections
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora)
model.print_trainable_parameters()   # shows the tiny % being trained

## Step 4: train

We tokenize, then hand everything to the `Trainer`. The data collator
builds language-modeling labels for us.

In [ ]:
def tokenize(batch):
    out = tokenizer(batch["text"], truncation=True, max_length=512)
    return out

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])

In [ ]:
from transformers import (
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
)

args = TrainingArguments(
    output_dir="./qwen-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch size 16
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

## Step 5: save and reuse the adapter

The adapter is only a few megabytes. Save it, then load it on top of the base
model whenever you need your fine-tuned behavior.

In [ ]:
model.save_pretrained("./qwen-lora-adapter")
tokenizer.save_pretrained("./qwen-lora-adapter")

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)
tuned = PeftModel.from_pretrained(base, "./qwen-lora-adapter")

prompt = "### Instruction:\nList two benefits of local SLMs.\n\n### Response:\n"
ids = tokenizer(prompt, return_tensors="pt").to(tuned.device)
out = tuned.generate(**ids, max_new_tokens=80)
print(tokenizer.decode(out[0], skip_special_tokens=True))

> **Tip.** To ship a single self-contained model, call
`tuned.merge_and_unload()` to fold the adapter back into the weights,
then `save_pretrained`. Keep the adapter separate while
experimenting; merge only when you deploy.

## Recap and exercise

You fine-tuned a base model on a domain dataset using LoRA, trained only a tiny
fraction of parameters, and saved a reusable adapter.

**Exercise.** Replace the dataset with 50–100 of your own
instruction/response pairs in a JSONL file and run one epoch. Compare the tuned
model's format-consistency against the base model on a held-out prompt.